# Llama Clean-K2 Matched Random-K Control (FP16 & INT4 NCALIB=128)

This notebook runs the matched random attention-head pair control ($K=2$, seed 42) for the Llama clean-K2 headline finding:
- **Random heads**: `L20_H24` and `L3_H19` (drawn with seed 42 from all 1024 heads, excluding the identified clean K2 pair `L20_H14`, `L26_H15` and the 10 context K10 heads).
- **Steering protocol**: Decode-time JuICE steering, $\alpha_+=1.0, \alpha_-=0.0$ (parametric enlarge direction), greedy decoding, `max_new_tokens=16`, on the preregistered $n=100$ conflict items.
- **Precisions**: Sequentially runs (a) FP16 base (`meta-llama/Llama-3.1-8B`) and (b) canonical INT4 (`llama31_8b_base_gptq_int4_ncalib128`).
- **Scoring**: Question-aware matcher + paired bootstrap (1000 resamples, seed 42) reporting both **inclusive $\Delta$** and **gold-PAR-only $\Delta$** matching Table 8.

### Recommended Kaggle Settings
- **Accelerator**: GPU T4 x2 (Free tier)
- **Internet**: On (for downloading FP16 weights / Hugging Face access)
- **Secrets**: Add `HF_TOKEN` with read access to `meta-llama/Llama-3.1-8B`
- **Datasets attached**: 
  1. `kaggle_llama_clean_k2_random_control.zip` (this package)
  2. `llama31_8b_base_gptq_int4_ncalib128` (canonical GPTQ INT4 model directory)


In [ ]:
# Setup environment, unpack package if attached as zip, and inspect GPUs
import os
import sys
import zipfile
from pathlib import Path
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    for d in range(torch.cuda.device_count()):
        print(f"  Device {d}: {torch.cuda.get_device_name(d)} ({(torch.cuda.get_device_properties(d).total_memory / 1e9):.1f} GB)")

# Unpack dataset if zip is present in /kaggle/input
for p in Path("/kaggle/input").rglob("*.zip") if Path("/kaggle/input").exists() else []:
    if "random_control" in str(p) or "llama" in str(p):
        print(f"Extracting {p} into /kaggle/working...")
        try:
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall("/kaggle/working")
        except Exception as e:
            print(f"Extract notice: {e}")

# HF Authentication via Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    hf_token = None
    for k in ['HF_TOKEN', 'hf_token', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
        try:
            hf_token = sec.get_secret(k)
            if hf_token:
                print(f"[AUTH] Found secret key '{k}' in Kaggle Secrets.")
                break
        except Exception:
            pass
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[AUTH] Successfully authenticated Hugging Face with Kaggle Secrets.')
    else:
        print('[AUTH] Warning: No HF_TOKEN secret found in Kaggle Secrets.')
except Exception as e:
    print(f'[AUTH] Kaggle Secrets login notice: {e}')

# Install gptqmodel for Kaggle Python 3.12 (required for INT4 GPTQ load)
try:
    import gptqmodel
except ImportError:
    import subprocess
    print('Installing gptqmodel==2.2.0...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gptqmodel==2.2.0', 'transformers==4.51.3', 'accelerate>=0.26.0,<2.0'])
    print('gptqmodel installed successfully.')

working = Path("/kaggle/working")
if str(working) not in sys.path:
    sys.path.insert(0, str(working))

# Find run script
candidates = list(working.rglob("run_llama_clean_k2_random_control.py"))
if Path("/kaggle/input").exists():
    candidates += list(Path("/kaggle/input").rglob("run_llama_clean_k2_random_control.py"))
if not candidates:
    candidates = list(Path(".").rglob("run_llama_clean_k2_random_control.py"))

assert candidates, "Could not find run_llama_clean_k2_random_control.py"
SCRIPT_PATH = candidates[0]
print(f"Run script resolved: {SCRIPT_PATH}")


In [ ]:
# Run Matched Random-K2 Control (FP16 + INT4 NCALIB=128)
import os
import subprocess

cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--fp16-model", "meta-llama/Llama-3.1-8B",
    "--seed", "42",
    "--n-boot", "1000",
]

print("Starting experiment run...")
env = os.environ.copy()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.stdout.close()
ret = proc.wait()
assert ret == 0, f"Experiment failed with returncode {ret}"


In [ ]:
# Display Table 8 Results and inspect generated output files
import pandas as pd
from IPython.display import display, Markdown

md_files = list(Path("/kaggle/working").rglob("llama_clean_k2_random_control_n100_ncalib128.md"))
csv_files = list(Path("/kaggle/working").rglob("llama_clean_k2_random_control_n100_ncalib128.csv"))

if md_files:
    content = md_files[0].read_text(encoding="utf-8")
    display(Markdown("### Table 8: Llama Clean-K2 Headline vs Matched Random-K2 Control"))
    display(Markdown(content))

if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df)

print("Done! You can download llama_clean_k2_random_control_n100_ncalib128.json, .md, and .csv from /kaggle/working.")
